# S10 · Inferencia en tiempo real con Spark MLlib

**Integrante** · Leonardo Jesus Huaman Arhuata

**Dimensión** · Volumen de aeronaves en vuelo

Equipo 04 · AeroStreamPeru · Ciclo 9

## 1. Etiqueta, predictores y selección del modelo

El notebook de [S5](../s05-evaluacion-unidad-1/05_evaluacion_leonardo.ipynb) construye la etiqueta con `lead()` y exige una ventana objetivo exactamente diez minutos después. Entrena con enero–agosto, valida en septiembre y prueba en octubre–noviembre. Calcula la persistencia antes de entrenar cinco candidatos, entre ellos una regresión lineal y un bosque aleatorio. Aquí leemos esa evidencia del contrato guardado.

In [1]:
INTEGRANTE = "leonardo"

from datetime import datetime, timedelta
from pathlib import Path
import json
import sys

SESIONES = Path("/opt")
S5 = SESIONES / "s05-evaluacion-unidad-1" / "artifacts"
S8 = SESIONES / "s08-streaming-estructurado"
sys.path.insert(0, str(S8))
from probar_s08 import leer_registros

contrato = json.loads((S5 / INTEGRANTE / "contrato_modelo_historico.json").read_text(encoding="utf-8"))
VARIABLE_OBJETIVO = contrato["variable_objetivo"]
VARIABLES_PREDICTORAS = [campo["nombre"] for campo in contrato["variables_predictoras"]]
print("Objetivo:", VARIABLE_OBJETIVO)
print("Predictoras:", VARIABLES_PREDICTORAS)
for periodo, resumen in contrato["periodos"].items():
    print(periodo, resumen["pares"], resumen["desde_utc"], resumen["hasta_utc"])

print(f"{'Candidato':<18} {'RMSE':>8} {'MAE':>8} {'R²':>8}")
for fila in sorted(contrato["validacion"], key=lambda registro: registro["rmse"]):
    print(f"{fila['modelo']:<18} {fila['rmse']:8.3f} {fila['mae']:8.3f} {fila['r2']:8.3f}")

print("Mejor estimador MLlib:", contrato["estimador_mllib"])
print("Preferido en validación:", contrato["preferido_en_validacion"])
print("Prueba reservada:", contrato["prueba"])

Objetivo: volumen_promedio_t1
Predictoras: ['volumen_promedio_t', 'volumen_min_t', 'volumen_max_t', 'velocidad_mediana_t', 'altitud_mediana_t', 'hora_del_dia', 'dia_semana']
entrenamiento 34535 2026-01-01T00:00:00 2026-08-31T23:50:00
validación 4319 2026-09-01T00:00:00 2026-09-30T23:50:00
prueba 7762 2026-10-01T00:00:00 2026-11-23T23:50:00
Candidato              RMSE      MAE       R²
Persistencia          2.704    2.116    0.746
Lineal base           2.803    2.221    0.728
Ridge                 2.841    2.249    0.720
Elastic Net           2.883    2.288    0.712
Lasso                 2.916    2.322    0.705
Bosque aleatorio      4.214    3.210    0.384
Mejor estimador MLlib: Lineal base
Preferido en validación: Persistencia
Prueba reservada: {'modelo': {'rmse': 2.2613583182087047, 'mae': 1.7616466726532467, 'r2': 0.885001997383754}, 'persistencia': {'rmse': 2.302714039552361, 'mae': 1.7933071373357388, 'r2': 0.8807573692060044}}


## 2. Evidencia de inferencia en streaming

[inferir_ventanas.py](inferir_ventanas.py) lee las ventanas de Kafka mediante `readStream`, carga el `PipelineModel` de S5 y aplica `transform()` a cada fila. La agregación y el watermark ocurren en S8. La consulta de inferencia recibe las predictoras ya calculadas y requiere al menos cuatro capturas.

In [2]:
MODO = "vivo"  # Usa "prueba" para leer las predicciones de la reproducción de S8.
if MODO == "prueba":
    evidencia = json.loads((S8 / "artifacts" / "evidencia_s08.json").read_text(encoding="utf-8"))
    prefijo = evidencia["topico_ventanas"].removesuffix(".ventanas.v1")
else:
    prefijo = "opensky"

topico = f"{prefijo}.predicciones.v1"
mensajes = leer_registros(topico, maximo_por_particion=100)
predicciones = {
    fila["prediction_id"]: fila for fila in mensajes if fila["integrante"] == INTEGRANTE
}
if not predicciones:
    raise RuntimeError("Todavía no hay predicciones. Verifica que Spark haya cerrado una ventana con al menos cuatro capturas.")

ultima = max(predicciones.values(), key=lambda fila: fila["inicio_utc"])
inicio = datetime.fromisoformat(ultima["inicio_utc"].replace("Z", "+00:00"))
objetivo = datetime.fromisoformat(ultima["inicio_objetivo_utc"].replace("Z", "+00:00"))
assert objetivo - inicio == timedelta(minutes=10)
assert ultima["capturas_observadas"] >= contrato["capturas_minimas"]
assert ultima["modelo"] == contrato["estimador_mllib"]
assert ultima["prediccion_persistencia"] == ultima["valor_actual"]
assert ultima["prediction_id"] == f"{ultima['window_id']}:{INTEGRANTE}"
print(json.dumps(ultima, indent=2, ensure_ascii=False))

{
  "window_id": "ventana:1795542000",
  "inicio_utc": "2026-11-24T17:40:00.000Z",
  "capturas_observadas": 4,
  "inicio_objetivo_utc": "2026-11-24T17:50:00.000Z",
  "fin_objetivo_utc": "2026-11-24T18:00:00.000Z",
  "integrante": "leonardo",
  "variable_objetivo": "volumen_promedio_t1",
  "unidad": "aeronaves",
  "modelo": "Lineal base",
  "valor_actual": 5.5,
  "prediccion_mllib": 5.545311434941987,
  "prediccion_persistencia": 5.5,
  "metodo_preferido": "Persistencia",
  "prediccion_preferida": 5.5,
  "version_esquema": 1,
  "prediction_id": "ventana:1795542000:leonardo"
}


## 3. Interpretación del resultado

La regresión lineal base se guarda como el mejor estimador MLlib. La persistencia tuvo menor RMSE en septiembre y se conserva como método preferido para el resultado operativo. Kafka publica ambas estimaciones para poder compararlas.

El `watermark` de S8 retrasa el cierre de la ventana de origen. El mensaje identifica el intervalo objetivo y registra la hora de emisión. Revisa esas horas al distinguir una corrida en vivo de una recuperación o una prueba histórica.